# Tutorial 2: Regression Analysis

Welcome to the second tutorial in our statistical learning series! In this notebook, we'll explore regression analysis - one of the most fundamental and widely used statistical techniques for understanding relationships between variables.

## Learning Objectives

By the end of this tutorial, you'll be able to:
- Understand and implement linear and logistic regression models
- Interpret regression coefficients and model diagnostics
- Assess model fit and make predictions
- Apply regression techniques to real-world problems

## 1. Setup and Introduction

Let's begin by importing the necessary libraries.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# Import our regression utilities
from statistics_lessons.ml_models.regression_examples import (
    linear_regression_example,
    logistic_regression_example
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Linear Regression: The Basics

Linear regression is a technique for modeling the relationship between a dependent variable (target) and one or more independent variables (features).

### 2.1 Simple Linear Regression

Let's start with a simple example to understand the basic concepts.

In [ ]:
# Generate synthetic data for simple linear regression
np.random.seed(42)
X = np.linspace(0, 10, 100)
y = 2 * X + 1 + np.random.normal(0, 1, 100)  # y = 2x + 1 + noise

# Convert to pandas Series for better visualization
data = pd.DataFrame({'X': X, 'y': y})

# Visualize the data
plt.scatter(X, y, alpha=0.7)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Simple Linear Regression Example')
plt.show()

Now, let's fit a linear regression model using our utility function.

In [ ]:
# Fit linear regression
slope, intercept = linear_regression_example(X, y)

# Print results
print(f"Estimated slope: {slope:.4f}")
print(f"Estimated intercept: {intercept:.4f}")
print(f"True slope: 2.0000")
print(f"True intercept: 1.0000")

# Plot the data and regression line
plt.scatter(X, y, alpha=0.7)
plt.plot(X, slope * X + intercept, 'r-', linewidth=2)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Simple Linear Regression')
plt.show()

### Interactive Exercise 1: Exploring Model Sensitivity

Let's explore how linear regression models are affected by outliers and sample size.

In [ ]:
# Function to generate data with outliers
def generate_data_with_outliers(slope, intercept, sample_size, outlier_ratio=0.1, outlier_strength=10, seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    # Generate regular data
    X = np.linspace(0, 10, sample_size)
    y = slope * X + intercept + np.random.normal(0, 1, sample_size)
    
    # Add outliers
    outlier_count = int(sample_size * outlier_ratio)
    outlier_idx = np.random.choice(sample_size, outlier_count, replace=False)
    outlier_direction = np.random.choice([-1, 1], outlier_count)
    y[outlier_idx] += outlier_direction * outlier_strength
    
    return X, y

# Function to visualize regression with metrics
def visualize_regression(X, y, title=None):
    slope, intercept = linear_regression_example(X, y)
    
    # Calculate metrics
    y_pred = slope * X + intercept
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(X, y, alpha=0.7)
    plt.plot(X, slope * X + intercept, 'r-', linewidth=2)
    plt.xlabel('X')
    plt.ylabel('y')
    
    if title:
        plt.title(title)
    else:
        plt.title(f'Linear Regression (Slope: {slope:.4f}, Intercept: {intercept:.4f})')
    
    plt.annotate(f'R²: {r2:.4f}\nRMSE: {rmse:.4f}', xy=(0.05, 0.95), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    plt.show()
    return slope, intercept, r2, rmse

Now, let's experiment with different scenarios:

In [ ]:
# Base case: No outliers, moderate sample size
X_base, y_base = generate_data_with_outliers(2, 1, 100, outlier_ratio=0, seed=42)
base_results = visualize_regression(X_base, y_base, 'Base Case: No Outliers')

# Case with outliers
X_outliers, y_outliers = generate_data_with_outliers(2, 1, 100, outlier_ratio=0.1, outlier_strength=10, seed=42)
outlier_results = visualize_regression(X_outliers, y_outliers, 'With 10% Strong Outliers')

# Small sample size
X_small, y_small = generate_data_with_outliers(2, 1, 20, outlier_ratio=0, seed=42)
small_results = visualize_regression(X_small, y_small, 'Small Sample Size (n=20)')

# Small sample with outliers
X_small_out, y_small_out = generate_data_with_outliers(2, 1, 20, outlier_ratio=0.1, outlier_strength=10, seed=42)
small_out_results = visualize_regression(X_small_out, y_small_out, 'Small Sample with Outliers')

🔍 **Based on the results above, answer these questions:**

1. How do outliers affect the estimated slope and intercept?
2. What impact do outliers have on R² and RMSE?
3. How does sample size influence the stability of parameter estimates?
4. Which scenario resulted in the most biased parameter estimates? Why?

<details>
<summary>Click for answers</summary>

1. Outliers can significantly pull the regression line toward them, biasing both slope and intercept estimates away from their true values.

2. Outliers typically reduce R² (worse fit) and increase RMSE (larger prediction errors).

3. Smaller sample sizes lead to less stable parameter estimates, as each data point has more influence on the model. The confidence intervals around the estimates would be wider with smaller samples.

4. The small sample with outliers scenario likely produced the most biased estimates because it combines the vulnerabilities of both small sample size and outlier influence. With fewer data points, each outlier has more leverage to pull the regression line.
</details>

## 3. Multiple Linear Regression

Let's move beyond simple linear regression and explore models with multiple predictors.

In [ ]:
# Generate synthetic data for multiple regression
np.random.seed(42)
n_samples = 200
X1 = np.random.normal(0, 1, n_samples)
X2 = np.random.normal(0, 1, n_samples)
X3 = np.random.normal(0, 1, n_samples)

# Create a target with known coefficients
y = 3*X1 + 1.5*X2 - 2*X3 + np.random.normal(0, 1, n_samples)

# Create a DataFrame
data_multi = pd.DataFrame({
    'X1': X1,
    'X2': X2,
    'X3': X3,
    'y': y
})

# Quick look at the data
data_multi.head()

Let's fit a multiple regression model using scikit-learn:

In [ ]:
from sklearn.linear_model import LinearRegression

# Prepare the feature matrix
X_multi = data_multi[['X1', 'X2', 'X3']]
y_multi = data_multi['y']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

# Fit the model
model = LinearRegression()
model.fit(X_train, y_train)

# Print the coefficients
print(f"Intercept: {model.intercept_:.4f}")
print("Coefficients:")
for feature, coef in zip(X_multi.columns, model.coef_):
    print(f"  {feature}: {coef:.4f} (True: {3 if feature == 'X1' else 1.5 if feature == 'X2' else -2})")

# Evaluate the model
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nModel Performance:")
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

### Visualizing Multiple Regression

In [ ]:
# Partial regression plots (also known as component-plus-residual plots)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(X_multi.columns):
    # Regress the feature against other features
    X_others = X_multi.drop(feature, axis=1)
    model_x = LinearRegression().fit(X_others, X_multi[feature])
    residuals_x = X_multi[feature] - model_x.predict(X_others)
    
    # Regress the target against other features
    model_y = LinearRegression().fit(X_others, y_multi)
    residuals_y = y_multi - model_y.predict(X_others)
    
    # Plot residuals against each other
    axes[i].scatter(residuals_x, residuals_y, alpha=0.7)
    
    # Add regression line to the plot
    slope, intercept = np.polyfit(residuals_x, residuals_y, 1)
    x_range = np.linspace(min(residuals_x), max(residuals_x), 100)
    axes[i].plot(x_range, slope * x_range + intercept, 'r-')
    
    axes[i].set_xlabel(f'Residuals for {feature}')
    axes[i].set_ylabel('Residuals for y')
    axes[i].set_title(f'Partial Regression Plot for {feature}')

plt.tight_layout()
plt.show()

### Interactive Exercise 2: Model Diagnostics and Assumptions

Let's explore linear regression assumptions and diagnostics.

In [ ]:
# Fit a model on the entire dataset
full_model = LinearRegression().fit(X_multi, y_multi)
y_pred_full = full_model.predict(X_multi)
residuals = y_multi - y_pred_full

# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs Fitted values
axes[0, 0].scatter(y_pred_full, residuals, alpha=0.7)
axes[0, 0].axhline(y=0, color='r', linestyle='-')
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# QQ Plot (normal probability plot of residuals)
from scipy import stats
sorted_residuals = np.sort(residuals)
theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_residuals)))
axes[0, 1].scatter(theoretical_quantiles, sorted_residuals, alpha=0.7)
axes[0, 1].plot(theoretical_quantiles, theoretical_quantiles, 'r-')
axes[0, 1].set_xlabel('Theoretical Quantiles')
axes[0, 1].set_ylabel('Sample Quantiles')
axes[0, 1].set_title('Normal Q-Q Plot')

# Scale-Location Plot (sqrt of standardized residuals vs fitted values)
standardized_residuals = residuals / np.std(residuals)
sqrt_abs_resid = np.sqrt(np.abs(standardized_residuals))
axes[1, 0].scatter(y_pred_full, sqrt_abs_resid, alpha=0.7)
axes[1, 0].set_xlabel('Fitted values')
axes[1, 0].set_ylabel('√|Standardized residuals|')
axes[1, 0].set_title('Scale-Location Plot')

# Residuals vs Leverage (Cook's distance)
from sklearn.metrics import pairwise_distances
leverage = np.diagonal(X_multi.values @ np.linalg.inv(X_multi.T.values @ X_multi.values) @ X_multi.T.values)
cook_dist = residuals**2 * leverage / (3 * (1 - leverage)**2)
axes[1, 1].scatter(leverage, standardized_residuals, s=cook_dist*500, alpha=0.7)
axes[1, 1].axhline(y=0, color='r', linestyle='-')
axes[1, 1].set_xlabel('Leverage')
axes[1, 1].set_ylabel('Standardized Residuals')
axes[1, 1].set_title("Residuals vs Leverage")

plt.tight_layout()
plt.show()

🔍 **Based on the diagnostic plots, answer these questions:**

1. Do the residuals appear to be normally distributed? How can you tell?
2. Is there evidence of heteroscedasticity (non-constant variance of residuals)?
3. Are there any high-leverage points that might be influencing the model?
4. Overall, do the linear regression assumptions appear to be satisfied?

<details>
<summary>Click for answers</summary>

1. The residuals appear to be normally distributed if the points in the Q-Q plot follow the diagonal line closely. Deviations from the line, especially at the tails, would suggest non-normality.

2. Heteroscedasticity would be evident in the Residuals vs Fitted and Scale-Location plots if there's a pattern or funnel shape. Ideally, there should be a random scatter of points with constant spread.

3. High-leverage points would appear in the Residuals vs Leverage plot in the upper or lower right corner. Points with large Cook's distance (larger circles) have more influence on the model.

4. In general, for this synthetic dataset, the assumptions should be reasonably well-satisfied since we generated the data to follow a linear model with normal errors. In real-world data, you might see more deviations from these assumptions.
</details>

## 4. Logistic Regression

Now let's move to logistic regression, which is used for binary classification problems.

In [ ]:
# Generate synthetic data for logistic regression
np.random.seed(42)
n_samples = 200
X_log = np.random.normal(0, 1, n_samples)

# Generate binary outcome with a sigmoid function
z = 3 * X_log + np.random.normal(0, 0.5, n_samples)  # Add noise
p = 1 / (1 + np.exp(-z))  # Convert to probability using sigmoid
y_log = (np.random.random(n_samples) < p).astype(int)  # Generate binary outcome

# Create a DataFrame
data_log = pd.DataFrame({
    'X': X_log,
    'y': y_log
})

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X_log, y_log, alpha=0.7)
plt.xlabel('X')
plt.ylabel('y (0 or 1)')
plt.title('Logistic Regression Example')
plt.show()

Now, let's fit a logistic regression model using our utility function.

In [ ]:
# Fit logistic regression
coef, intercept = logistic_regression_example(X_log, y_log)

# Print results
print(f"Estimated coefficient: {coef:.4f}")
print(f"Estimated intercept: {intercept:.4f}")

# Create a grid for prediction curve
X_grid = np.linspace(min(X_log), max(X_log), 100)
# Convert log-odds to probabilities using the sigmoid function
z_grid = coef * X_grid + intercept
p_grid = 1 / (1 + np.exp(-z_grid))

# Plot the data and predicted probabilities
plt.figure(figsize=(10, 6))
plt.scatter(X_log, y_log, alpha=0.7)
plt.plot(X_grid, p_grid, 'r-', linewidth=2)
plt.xlabel('X')
plt.ylabel('Probability of y=1')
plt.title('Logistic Regression')
plt.show()

### Understanding Logistic Regression Coefficients

Unlike linear regression, the coefficients in logistic regression represent the change in log-odds for a one-unit increase in the predictor.

In [ ]:
# Interpret coefficients
print("Logistic Regression Coefficient Interpretation:")
print(f"For each one-unit increase in X, the log-odds of y=1 increase by {coef:.4f}")
print(f"This translates to an odds ratio of {np.exp(coef):.4f}")
print(f"For example, if X increases from 0 to 1:")
print(f"  - The odds of y=1 are multiplied by {np.exp(coef):.4f}")
x0 = 0
x1 = 1
p0 = 1 / (1 + np.exp(-(coef * x0 + intercept)))
p1 = 1 / (1 + np.exp(-(coef * x1 + intercept)))
print(f"  - The probability changes from {p0:.4f} to {p1:.4f}")

### Interactive Exercise 3: Threshold Selection and Model Evaluation

In logistic regression, we need to choose a probability threshold to convert predicted probabilities to binary predictions.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Function to evaluate logistic regression at different thresholds
def evaluate_threshold(X, y, threshold=0.5):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    
    # Reshape X for sklearn
    X_reshaped = X.reshape(-1, 1)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.3, random_state=42)
    
    # Fit model
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    
    # Get probabilities
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Convert to class predictions based on threshold
    y_pred = (y_proba >= threshold).astype(int)
    
    # Compute metrics
    conf_matrix = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = conf_matrix.ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'threshold': threshold,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'conf_matrix': conf_matrix,
        'y_test': y_test,
        'y_proba': y_proba,
        'y_pred': y_pred
    }

# Evaluate multiple thresholds
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
results = []

for threshold in thresholds:
    result = evaluate_threshold(X_log, y_log, threshold)
    results.append(result)
    
    print(f"\nThreshold: {threshold}")
    print(f"Accuracy: {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall: {result['recall']:.4f}")
    print(f"F1 Score: {result['f1']:.4f}")
    print("Confusion Matrix:")
    print(result['conf_matrix'])

Let's visualize how different thresholds affect our predictions:

In [ ]:
# Plot ROC curve
plt.figure(figsize=(10, 6))

# Use the last evaluation result
fpr, tpr, roc_thresholds = roc_curve(results[-1]['y_test'], results[-1]['y_proba'])
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)

# Mark the evaluated thresholds on the ROC curve
for result in results:
    threshold = result['threshold']
    y_test = result['y_test']
    y_pred = result['y_pred']
    
    # Calculate false positive rate and true positive rate for this threshold
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    tpr_val = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    plt.plot(fpr_val, tpr_val, 'ro', markersize=8)
    plt.text(fpr_val+0.02, tpr_val-0.02, f"{threshold}", fontsize=9)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# Plot metrics vs threshold
metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_values = {metric: [result[metric] for result in results] for metric in metrics}

plt.figure(figsize=(10, 6))
for metric, values in metric_values.items():
    plt.plot(thresholds, values, 'o-', label=metric)

plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Threshold')
plt.legend()
plt.grid(True)
plt.show()

🔍 **Based on the threshold analysis, answer these questions:**

1. How does changing the threshold affect precision and recall?
2. What threshold would you choose if false positives were very costly?
3. What threshold maximizes the F1 score (balance of precision and recall)?
4. In what real-world scenarios might you prioritize precision over recall, or vice versa?

<details>
<summary>Click for answers</summary>

1. As the threshold increases, precision typically increases (fewer false positives) while recall decreases (more false negatives). This represents the precision-recall trade-off.

2. If false positives are very costly, you would choose a higher threshold (like 0.7 or 0.9) to minimize them, accepting that you'll miss some positive cases.

3. The threshold that maximizes F1 score is usually somewhere in the middle, often around 0.3-0.5 for balanced datasets. The exact value depends on the dataset.

4. Scenarios where precision is prioritized:
   - Spam detection (false positives would mean legitimate emails get filtered)
   - Medical screening where follow-up tests are invasive or expensive
   
   Scenarios where recall is prioritized:
   - Cancer detection (missing a cancer case is worse than a false alarm)
   - Fraud detection (missing fraud is more costly than investigating a false positive)
   - Predictive maintenance (missing equipment failure is very costly)
</details>

## 5. Multiple Logistic Regression

Let's extend our logistic regression model to include multiple predictors.

In [ ]:
# Generate synthetic data for multiple logistic regression
np.random.seed(42)
n_samples = 500
X1 = np.random.normal(0, 1, n_samples)
X2 = np.random.normal(0, 1, n_samples)
X3 = np.random.normal(0, 1, n_samples)

# Generate binary outcome
z = 2*X1 - 1*X2 + 0.5*X3 + np.random.normal(0, 0.5, n_samples)
p = 1 / (1 + np.exp(-z))
y = (np.random.random(n_samples) < p).astype(int)

# Create a DataFrame
data_multi_log = pd.DataFrame({
    'X1': X1,
    'X2': X2,
    'X3': X3,
    'y': y
})

# Fit multiple logistic regression
from sklearn.linear_model import LogisticRegression

X_multi_log = data_multi_log[['X1', 'X2', 'X3']]
y_multi_log = data_multi_log['y']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_multi_log, y_multi_log, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the model
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Print coefficients
print("Logistic Regression Coefficients:")
for feature, coef in zip(X_multi_log.columns, log_reg.coef_[0]):
    print(f"  {feature}: {coef:.4f} (True: {2 if feature == 'X1' else -1 if feature == 'X2' else 0.5})")
print(f"Intercept: {log_reg.intercept_[0]:.4f}")

# Evaluate the model
y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Multiple Logistic Regression')
plt.legend(loc="lower right")
plt.show()

## 6. Feature Importance and Interpretation

Understanding which features are most important is crucial for model interpretation.

In [ ]:
# Visualize feature importance for logistic regression
feature_importance = pd.DataFrame({
    'Feature': X_multi_log.columns,
    'Absolute Coefficient': np.abs(log_reg.coef_[0]),
    'Coefficient': log_reg.coef_[0]
})
feature_importance = feature_importance.sort_values('Absolute Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'])
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance in Logistic Regression')
plt.grid(axis='x')
plt.show()

# Odds ratios
print("\nOdds Ratios (effect of one std. dev. increase):")
for feature, coef in zip(X_multi_log.columns, log_reg.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"  {feature}: {odds_ratio:.4f} times higher odds of y=1")

## 7. Practice Exercise: Real-World Regression Analysis

Now it's your turn to apply regression techniques to a real dataset!

In [ ]:
# Load the California housing dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X_housing = housing.data
y_housing = housing.target

print("Dataset information:")
print(f"Number of samples: {X_housing.shape[0]}")
print(f"Number of features: {X_housing.shape[1]}")
print("\nFeatures:")
for feature in X_housing.columns:
    print(f"  {feature}")
print("\nTarget: Median house value ($100,000s)")

# Your task: Build both linear and logistic regression models
# 1. For linear regression, predict the house values
# 2. For logistic regression, create a binary target (e.g., high value if >median)
# 3. Explore feature importance and interpret the coefficients
# 4. Evaluate model performance
# 5. Create diagnostic plots

<details>
<summary>Click for sample solution</summary>

In [ ]:
# Exploratory data analysis
X_housing.describe()

# Visualize feature distributions
X_housing.hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()

# Correlation matrix
plt.figure(figsize=(12, 10))
correlation_matrix = pd.concat([X_housing, y_housing], axis=1).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlations')
plt.show()

# Linear Regression
# Split data
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Standardize features
scaler_lin = StandardScaler()
X_train_lin_scaled = scaler_lin.fit_transform(X_train_lin)
X_test_lin_scaled = scaler_lin.transform(X_test_lin)

# Fit model
linear_model = LinearRegression()
linear_model.fit(X_train_lin_scaled, y_train_lin)

# Coefficients
coef_df = pd.DataFrame({
    'Feature': X_housing.columns,
    'Coefficient': linear_model.coef_
})
coef_df = coef_df.sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance in Linear Regression')
plt.grid(axis='x')
plt.show()

# Predictions and evaluation
y_pred_lin = linear_model.predict(X_test_lin_scaled)
mse = mean_squared_error(y_test_lin, y_pred_lin)
r2 = r2_score(y_test_lin, y_pred_lin)

print(f"Linear Regression Results:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R²: {r2:.4f}")
print(f"RMSE: {np.sqrt(mse):.4f}")

# Residual plot
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_lin, y_test_lin - y_pred_lin, alpha=0.7)
plt.axhline(y=0, color='r', linestyle='-')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()

# Logistic Regression
# Create binary target (above median = 1, below median = 0)
median_value = y_housing.median()
y_binary = (y_housing > median_value).astype(int)

# Split data
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_housing, y_binary, test_size=0.2, random_state=42
)

# Standardize features
scaler_log = StandardScaler()
X_train_log_scaled = scaler_log.fit_transform(X_train_log)
X_test_log_scaled = scaler_log.transform(X_test_log)

# Fit model
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_log_scaled, y_train_log)

# Coefficients
log_coef_df = pd.DataFrame({
    'Feature': X_housing.columns,
    'Coefficient': log_model.coef_[0],
    'Odds Ratio': np.exp(log_model.coef_[0])
})
log_coef_df = log_coef_df.sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(log_coef_df['Feature'], log_coef_df['Coefficient'])
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance in Logistic Regression')
plt.grid(axis='x')
plt.show()

# Predictions and evaluation
y_pred_log = log_model.predict(X_test_log_scaled)
y_proba_log = log_model.predict_proba(X_test_log_scaled)[:, 1]

print("\nLogistic Regression Results:")
print(classification_report(y_test_log, y_pred_log))

# ROC curve
fpr, tpr, _ = roc_curve(y_test_log, y_proba_log)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

# Confusion matrix
plt.figure(figsize=(8, 6))
conf_mat = confusion_matrix(y_test_log, y_pred_log)
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.show()

</details>

## 8. Summary and Key Takeaways

In this tutorial, we've covered:

1. **Simple linear regression** - modeling relationships between two variables
2. **Multiple linear regression** - incorporating multiple predictors
3. **Regression diagnostics** - checking model assumptions and fit
4. **Logistic regression** - modeling binary outcomes
5. **Threshold selection** - balancing precision and recall
6. **Model interpretation** - understanding coefficients and feature importance

### Next Steps

In the next tutorial, we'll explore:
- Hypothesis testing and inference
- Confidence intervals and p-values
- Techniques for comparing groups
- Power analysis and sample size determination

## 9. Additional Resources

- [An Introduction to Statistical Learning](https://www.statlearning.com/)
- [Scikit-learn Documentation: Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
- [Statsmodels Documentation](https://www.statsmodels.org/stable/index.html)
- [The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/)
- [Linear Regression in Python](https://realpython.com/linear-regression-in-python/)